# TokenMLP under Protocol A (about 1.5–2.5 h on a T4 GPU)
**Purpose.** In the paper, the parameter-matched attention-free control (TokenMLP) was trained only on the source-group partitions
(Protocol B). This notebook trains it on the five **image-level partitions (Protocol A, seeds 42–46)** with exactly the same pipeline,
partitions and settings as the Protocol A models, so that the hybrid model can be compared with its parameter-matched control under
both protocols.

**How to run (Google Colab):** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Allow Google Drive access.
Results are saved to `MyDrive/tokenmlp_protocolA_results`. **If Colab disconnects, run all cells again** – finished seeds are skipped.
At the end a SUMMARY is printed and `tokenmlp_protocolA_results.zip` is downloaded. **Send both to Claude.**

If the earlier Protocol A results are still in `MyDrive/extra_results/`, the notebook checks that the partitions are identical and
prints paired differences with the hybrid model and EfficientNetB0. Nothing else is needed.

In [ ]:
# %% Settings
import os
TEST_MODE = os.environ.get("GS_TEST_MODE") == "1"     # offline test only; leave as is
SEEDS           = [42, 43, 44, 45, 46]
IMG_SIZE        = 160
BATCH_SIZE      = 16
EPOCHS_STAGE1   = 8
EPOCHS_STAGE2   = 8
LR_STAGE1       = 1e-4
LR_STAGE2       = 5e-5
UNFREEZE_LAST_N = 40
PATIENCE        = 5
DROPOUT         = 0.3
WEIGHTS         = "imagenet"
TILE            = 400
N_BOOT          = 2000
FM_NAME         = "owkin/phikon"                  # public pathology foundation model (Hugging Face)
FM_BATCH        = 64
GROUPS_COMMIT = "912f342fe2a781e847f395d79982dad4d3d425ae"
GROUPS_SHA256 = "2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23"
GROUPS_URL = f"https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/{GROUPS_COMMIT}/kaggle/lc25000_image_groups.csv"
LH_URL     = "https://ndownloader.figshare.com/files/45206104"
SAVE_DIR   = "/content/drive/MyDrive/tokenmlp_protocolA_results"
REF_DIR    = "/content/drive/MyDrive/extra_results"            # earlier Protocol A run (optional, for checks)
GS_DIR     = "/content/drive/MyDrive/group_split_results"        # results of the group-split notebook (for paired comparison)
DATA_DIR   = None
LH_ROOT    = "/content/lunghist700"
CNN_NAMES  = ["EffNetB0_TokenMLP"]
if TEST_MODE:
    EPOCHS_STAGE1, EPOCHS_STAGE2, WEIGHTS, N_BOOT, SEEDS, FM_BATCH = 1, 1, None, 100, [42, 43], 8
    DATA_DIR, LH_ROOT, SAVE_DIR = os.environ["GS_DATA"], os.environ["GS_LH"], os.environ["GS_OUT"]
    GS_DIR = os.environ.get("GS_GSDIR", "/nonexistent"); REF_DIR = os.environ.get("GS_REF", "/nonexistent")
    GROUPS_URL = os.environ["GS_GROUPS"]

In [ ]:
# %% Imports and shared helpers
import json, time, hashlib, random, glob, re, subprocess, zipfile, platform
import numpy as np, pandas as pd, cv2
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

def ece_score(p, y, n_bins=15):
    conf, pred = p.max(1), p.argmax(1); bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def cluster_ci(correct, clusters, seed=0):
    rng = np.random.default_rng(seed); u = np.unique(clusters); idx = {k: np.where(clusters == k)[0] for k in u}; a = []
    for _ in range(N_BOOT):
        s = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)]); a.append(correct[s].mean())
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def metrics(p, y, clusters):
    yp = p.argmax(1); cm = confusion_matrix(y, yp, labels=[0, 1, 2]); rec = cm.diagonal() / cm.sum(1).clip(min=1)
    return {"n": int(len(y)), "accuracy": float((yp == y).mean()),
            "accuracy_95ci_cluster": cluster_ci((yp == y).astype(float), clusters),
            "macro_f1": float(f1_score(y, yp, average="macro")), "balanced_accuracy": float(balanced_accuracy_score(y, yp)),
            "ece": ece_score(p, y), "brier": float(((p - np.eye(3)[y]) ** 2).sum(1).mean()),
            "recall": dict(zip(CLASSES, rec.round(4).tolist())),
            "auc_ovr_macro": float(roc_auc_score(y, p, multi_class="ovr")) if len(np.unique(y)) == 3 else None,
            "errors": int((yp != y).sum()), "confusion_matrix": cm.tolist()}

def agg(vals):
    v = np.array(vals, float); m = v.mean(); sd = v.std(ddof=1) if len(v) > 1 else 0.0
    return {"mean": float(m), "sd": float(sd), "min": float(v.min()), "max": float(v.max()), "per_seed": v.round(5).tolist()}

In [ ]:
# %% LC25000 (MD5-deduplicated) + source-tile groups + both partitioning protocols (identical to the earlier notebooks)
def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
    raise FileNotFoundError(root)
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
else:
    DATA_DIR = find_lung_dir(DATA_DIR)
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "filename": os.path.basename(f), "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
groups = pd.read_csv(GROUPS_URL)
if not TEST_MODE:
    import urllib.request
    assert hashlib.sha256(urllib.request.urlopen(GROUPS_URL).read()).hexdigest() == GROUPS_SHA256, "grouping file changed - send this message to Claude"
groups = groups[groups["label"].isin(CLASSES)][["filename", "label", "group_id"]].rename(columns={"label": "g_label"})
df = df.merge(groups, on="filename", how="left")
assert df["group_id"].notna().all(), "images without group - send this message to Claude"
assert (df["g_label"] == df["cls"]).all()
df["group_id"] = df["group_id"].astype(int)
print(f"LC25000 lung images kept: {len(df)} | groups: {df['group_id'].nunique()}")

def image_split(df, seed):
    """Protocol A: conventional stratified image-level 70/15/15 split (seed 42 reproduces the first benchmark)."""
    tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=seed)
    parts = {"train": tr.reset_index(drop=True), "val": va.reset_index(drop=True), "test": te.reset_index(drop=True)}
    for a in parts:
        for b in parts:
            if a < b: assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

def group_split(df, seed, fr=(0.70, 0.15, 0.15)):
    """Protocol B: identical to the group-split notebook (whole source groups, largest relative shortfall)."""
    rng = np.random.default_rng(seed); assign = {}; keys = ["train", "val", "test"]
    for c in CLASSES:
        sizes = df[df["cls"] == c].groupby("group_id").size()
        g = sizes.index.values.copy(); rng.shuffle(g)
        target = np.array(fr) * sizes.sum(); filled = np.zeros(3)
        for gid in g:
            k = int(np.argmax((target - filled) / target)); assign[gid] = keys[k]; filled[k] += sizes[gid]
    s = df.assign(split=df["group_id"].map(assign))
    parts = {k: s[s["split"] == k].reset_index(drop=True) for k in keys}
    for a in parts:
        for b in parts:
            if a < b:
                assert not set(parts[a]["group_id"]) & set(parts[b]["group_id"]), "group overlap!"
                assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

SPLITS = {("A", s): image_split(df, s) for s in SEEDS}
SPLITS.update({("B", s): group_split(df, s) for s in SEEDS})
leak = {s: float(100 * SPLITS[("A", s)]["test"]["group_id"].isin(set(SPLITS[("A", s)]["train"]["group_id"])).mean()) for s in SEEDS}
print("Protocol A leakage (% of test images whose source group is in training):", leak)
for (p, s), parts in SPLITS.items():
    d = os.path.join(SAVE_DIR, f"splits/protocol{p}_seed{s}"); os.makedirs(d, exist_ok=True)
    for k, v in parts.items(): v[["filename", "label", "group_id"]].to_csv(os.path.join(d, f"{k}.csv"), index=False)

# Check that the Protocol A partitions are identical to those of the paper (if the earlier run is on Drive)
n_checked = 0
for s in SEEDS:
    for k in ["train", "val", "test"]:
        f = os.path.join(REF_DIR, f"splits/protocolA_seed{s}/{k}.csv")
        if os.path.exists(f):
            assert sorted(pd.read_csv(f)["filename"]) == sorted(SPLITS[("A", s)][k]["filename"]), f"partition differs: seed {s} {k} - send this message to Claude"
            n_checked += 1
print(f"Protocol A partitions checked against the earlier run: {n_checked} of {3 * len(SEEDS)} files identical"
      + ("" if n_checked else " (earlier splits not found - partitions are regenerated with the same code and seeds)"))

In [ ]:
# %% LungHist700 index (external only; no tuning)
if not glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True):
    os.makedirs(LH_ROOT, exist_ok=True); rar = "/content/LungHist700.rar"
    if not os.path.exists(rar): subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if subprocess.run(["unrar", "x", "-o+", "-inul", rar, LH_ROOT + "/"]).returncode != 0:
        subprocess.run(["7z", "x", "-y", f"-o{LH_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)
rows = []
for f in sorted(glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True)):
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", os.path.basename(f).lower())
    if m:
        sup, sub, res_, iid = m.groups()
        rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res_, "image_id": int(iid),
                     "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
for c in glob.glob(LH_ROOT + "/**/*.csv", recursive=True):
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("").astype(str).str.lower().replace("nan", "")
        meta["superclass"] = meta["superclass"].astype(str).str.lower(); meta["resolution"] = meta["resolution"].astype(str).str.lower()
        meta["image_id"] = meta["image_id"].astype(int)
        ext = ext.merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                        on=["superclass", "subclass", "resolution", "image_id"], how="left"); break
assert "patient_id" in ext and ext["patient_id"].notna().all(), "patient IDs missing - send this message to Claude"
y_ext, pat = ext["label"].values, ext["patient_id"].values
T = 12
print(f"LungHist700: {len(ext)} images, {ext['patient_id'].nunique()} patients")

def ext_tiles(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    out = [img[yy:yy + TILE, xx:xx + TILE] for yy in range(0, H - TILE + 1, TILE) for xx in range(0, W - TILE + 1, TILE)]
    assert len(out) == T, f"{path}: {len(out)} tiles instead of 12 - send this message to Claude"
    return out

def ext_result(p):
    res = metrics(p, y_ext, pat)
    res["by_resolution"] = {r: float((p.argmax(1)[ext["resolution"] == r] == y_ext[ext["resolution"] == r]).mean()) for r in ["20x", "40x"]}
    pp = ext[["patient_id", "label"]].assign(**{f"p{i}": p[:, i] for i in range(3)}).groupby(["patient_id", "label"], as_index=False).mean()
    res["patient_class_accuracy"] = float((pp[["p0", "p1", "p2"]].to_numpy().argmax(1) == pp["label"].to_numpy()).mean())
    res["patient_class_units"] = int(len(pp))
    return res

In [ ]:
# %% Train TokenMLP on the Protocol A partitions (same pipeline and settings as the Protocol A CNNs)
import tensorflow as tf, keras
from keras import layers
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass
keras.mixed_precision.set_global_policy("float32")
AUTOTUNE = tf.data.AUTOTUNE
try:
    _gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() if not TEST_MODE else "test"
except FileNotFoundError:
    _gpu = "none (CPU runtime)"
json.dump({"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__, "gpu": _gpu, "seeds": SEEDS,
           "model": "EffNetB0_TokenMLP", "protocolA": "stratified image-level 70/15/15, seeds 42-46"},
          open(os.path.join(SAVE_DIR, "config.json"), "w"), indent=1)

def seed_all(s):
    os.environ["PYTHONHASHSEED"] = str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
def make_augment(seed):
    return keras.Sequential([layers.RandomFlip("horizontal", seed=seed + 101), layers.RandomRotation(15 / 360, seed=seed + 102),
                             layers.RandomTranslation(0.1, 0.1, seed=seed + 103), layers.RandomZoom(0.1, seed=seed + 104)], name=f"augment_seed{seed}")
def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8), tf.one_hot(label, len(CLASSES))
def make_ds(part, training, seed):
    ds = tf.data.Dataset.from_tensor_slices((part["path"].values, part["label"].values)).map(load, num_parallel_calls=AUTOTUNE).cache()
    if training: ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        aug = make_augment(seed); ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)
def head(x):
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    return layers.Activation("softmax", name="pred")(layers.Dense(len(CLASSES), name="logits")(x))
def build_mobilenet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image"); x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)
    base = keras.applications.MobileNetV2(include_top=False, weights=WEIGHTS, input_tensor=x)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="MobileNetV2"), base
def build_efficientnet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="EfficientNetB0"), base
def build_hybrid():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    fmap = base.output; h, w, c = fmap.shape[1], fmap.shape[2], fmap.shape[3]
    tokens = layers.Reshape((h * w, c), name="tokens")(fmap)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tokens)
    x = layers.Add(name="res1")([tokens, layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(t, t)])
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
    t = layers.Dense(c, name="mlp2")(layers.Dense(128, activation="gelu", name="mlp1")(t))
    x = layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([x, t]))
    return keras.Model(inp, head(x), name="Hybrid_EffNetB0_Transformer"), base
def build_tokenmlp():
    """Ablation (identical to notebook 4): the hybrid block with self-attention removed and the token MLP widened (hidden 192)
    so that the added parameters match the transformer block (~0.50M)."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    fmap = base.output; h, w, c = fmap.shape[1], fmap.shape[2], fmap.shape[3]
    tokens = layers.Reshape((h * w, c), name="tokens")(fmap)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(tokens)
    t = layers.Dense(c, name="mlp2")(layers.Dense(192, activation="gelu", name="mlp1")(t))
    x = layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([tokens, t]))
    return keras.Model(inp, head(x), name="EffNetB0_TokenMLP"), base
BUILDERS = {"MobileNetV2": build_mobilenet, "EfficientNetB0": build_efficientnet, "Hybrid_EffNetB0_Transformer": build_hybrid, "EffNetB0_TokenMLP": build_tokenmlp}
def set_stage2(base):
    base.trainable = True
    for l in base.layers[:-UNFREEZE_LAST_N]: l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False
def callbacks(ckpt):
    return [keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True, save_weights_only=True),
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

tiles_cache = None
for seed in SEEDS:
    parts = SPLITS[("A", seed)]; sdir = os.path.join(SAVE_DIR, "protocolA_cnn", f"seed{seed}"); os.makedirs(sdir, exist_ok=True)
    print(f"\nPROTOCOL A, SEED {seed}: " + " | ".join(f"{k} {len(p)}" for k, p in parts.items()) + f" | leakage {leak[seed]:.1f}%")
    for name in CNN_NAMES:
        mdir = os.path.join(sdir, name); done = os.path.join(mdir, "internal.json"); edone = os.path.join(mdir, "external.json")
        if os.path.exists(done) and os.path.exists(edone):
            print(f"  {name}: already finished - skipped"); continue
        os.makedirs(mdir, exist_ok=True); keras.backend.clear_session(); seed_all(seed)
        model, base = BUILDERS[name](); ckpt = os.path.join(mdir, "best.weights.h5")
        if not os.path.exists(done):
            print("=" * 70 + f"\nPROTOCOL A, SEED {seed} - TRAINING {name}\n" + "=" * 70)
            tr, va, te = make_ds(parts["train"], True, seed), make_ds(parts["val"], False, seed), make_ds(parts["test"], False, seed)
            t0 = time.time(); base.trainable = False
            model.compile(keras.optimizers.Adam(LR_STAGE1), "categorical_crossentropy", metrics=["accuracy"])
            h1 = model.fit(tr, validation_data=va, epochs=EPOCHS_STAGE1, callbacks=callbacks(ckpt), verbose=2)
            set_stage2(base); model.compile(keras.optimizers.Adam(LR_STAGE2), "categorical_crossentropy", metrics=["accuracy"])
            h2 = model.fit(tr, validation_data=va, epochs=EPOCHS_STAGE2, callbacks=callbacks(ckpt), verbose=2)
            model.load_weights(ckpt)
            hist = {k: [float(v) for v in h1.history[k] + h2.history[k]] for k in h1.history if k in h2.history}
            p = model.predict(te, verbose=0); y = parts["test"]["label"].values
            res = metrics(p, y, parts["test"]["group_id"].values)
            res.update({"seed": seed, "model": name, "protocol": "A", "minutes": round((time.time() - t0) / 60, 1),
                        "params_total": int(model.count_params()),
                        "best_val_accuracy_stage1": float(max(hist["val_accuracy"][:len(h1.history["loss"])]))})
            pd.DataFrame({"filename": parts["test"]["filename"], "group_id": parts["test"]["group_id"], "y_true": y, "y_pred": p.argmax(1),
                          **{f"p_{c}": p[:, i] for i, c in enumerate(CLASSES)}}).to_csv(os.path.join(mdir, "test_predictions.csv"), index=False)
            json.dump(hist, open(os.path.join(mdir, "history.json"), "w")); json.dump(res, open(done, "w"), indent=1)
            print(f"  -> {name} Protocol A seed {seed}: accuracy {100*res['accuracy']:.2f}% | macro-F1 {res['macro_f1']:.4f} | {res['minutes']} min")
        else:
            model.load_weights(ckpt)
        if tiles_cache is None:
            tiles_cache = np.stack([cv2.resize(t, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) for f in ext["path"] for t in ext_tiles(f)]).astype(np.float32)
        pe = model.predict(tiles_cache, batch_size=64, verbose=0).reshape(len(ext), T, 3).mean(1)
        np.save(os.path.join(mdir, "external_probs.npy"), pe); json.dump(ext_result(pe), open(edone, "w"), indent=1)
        print(f"  -> {name} Protocol A seed {seed}: external accuracy {100*json.load(open(edone))['accuracy']:.1f}%")
        del model

In [ ]:
# %% SUMMARY (copy everything printed below and send it to Claude together with tokenmlp_protocolA_results.zip)
from statsmodels.stats.contingency_tables import mcnemar
name = "EffNetB0_TokenMLP"; S = {"protocolA_leakage_pct": leak, "tokenmlp_A": {}, "paired": {}}
RI = [json.load(open(os.path.join(SAVE_DIR, "protocolA_cnn", f"seed{s}", name, "internal.json"))) for s in SEEDS]
RE = [json.load(open(os.path.join(SAVE_DIR, "protocolA_cnn", f"seed{s}", name, "external.json"))) for s in SEEDS]
S["tokenmlp_A"] = {**{f"int_{k}": agg([r[k] for r in RI]) for k in ["accuracy", "macro_f1", "balanced_accuracy", "ece", "brier"]},
                   **{f"ext_{k}": agg([r[k] for r in RE]) for k in ["accuracy", "macro_f1", "balanced_accuracy", "ece"]},
                   "int_errors": [r["errors"] for r in RI], "int_best_val_accuracy_stage1": agg([r["best_val_accuracy_stage1"] for r in RI]),
                   "minutes": [r["minutes"] for r in RI], "params_total": RI[0].get("params_total")}
for other in ["Hybrid_EffNetB0_Transformer", "EfficientNetB0", "MobileNetV2"]:
    rows = []
    for s in SEEDS:
        fo = os.path.join(REF_DIR, "protocolA_cnn", f"seed{s}", other, "test_predictions.csv")
        if not os.path.exists(fo): break
        ta = pd.read_csv(os.path.join(SAVE_DIR, "protocolA_cnn", f"seed{s}", name, "test_predictions.csv")); tb = pd.read_csv(fo)
        assert (ta["filename"].values == tb["filename"].values).all(), "test order differs - send this message to Claude"
        ca, cb = (ta.y_pred == ta.y_true).values, (tb.y_pred == tb.y_true).values
        tab = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
        rows.append({"seed": s, "diff_pp": float(100 * (ca.mean() - cb.mean())), "mcnemar_p": float(mcnemar(tab, exact=True).pvalue)})
    if len(rows) == len(SEEDS): S["paired"][f"TokenMLP - {other}"] = rows
json.dump(S, open(os.path.join(SAVE_DIR, "tokenmlp_protocolA_summary.json"), "w"), indent=1)

def f(a, sc=100, d=2): return f"{sc*a['mean']:.{d}f} ± {sc*a['sd']:.{d}f}"
m = S["tokenmlp_A"]
print("=" * 78 + "\nSUMMARY - TokenMLP under Protocol A, seeds", SEEDS)
print("Protocol A leakage (%):", leak)
print(f"TokenMLP Protocol A: internal {f(m['int_accuracy'])}% per seed {[round(100*r['accuracy'], 2) for r in RI]} | errors {m['int_errors']}")
print(f"   macro-F1 {f(m['int_macro_f1'],1,4)} | ECE {f(m['int_ece'],1,4)} | Stage-1 best val {f(m['int_best_val_accuracy_stage1'])}% | minutes {m['minutes']} | params {m['params_total']}")
print(f"   external {f(m['ext_accuracy'],100,1)}% per seed {[round(100*r['accuracy'], 1) for r in RE]} | balanced {f(m['ext_balanced_accuracy'],100,1)}%")
print("   external recall aca/nor/scc per seed:", [[round(r['recall'][c], 2) for c in CLASSES] for r in RE])
for k, rows in S["paired"].items():
    print(f"   {k} (internal, pp) per seed: " + "; ".join(f"{r['diff_pp']:+.2f} (McNemar p {r['mcnemar_p']:.3f})" for r in rows))
if not S["paired"]: print("   (earlier Protocol A predictions not found on Drive - paired differences will be computed by Claude)")
print("=" * 78)
if not TEST_MODE:
    subprocess.run(f"cd {os.path.dirname(SAVE_DIR)} && zip -qr /content/tokenmlp_protocolA_results.zip {os.path.basename(SAVE_DIR)} -x '*.h5'", shell=True)
    from google.colab import files; files.download("/content/tokenmlp_protocolA_results.zip")